# Unified Script-Aware Hybrid CTC/Attention HTR x AraMS-28k — Kaggle Training

**Architecture:** Asymmetric-stride `ResNet-34` + Masked Style/Script `FiLM` + Multi-Scale + Vertical Attn Pool + 2-layer BiLSTM (`768`d) + Aux 38-class CTC Head + Pretrained `hatformer-muharaf` `TrOCRForCausalLM` decoder.

**Setup (Kaggle sidebar -> Session options):**
1. Accelerator = **GPU T4 x2** (uses 1 GPU cleanly; avoids P100).
2. **Internet = ON**.
3. Run once with `SMOKE_TEST = True` (~5 min Stage 1 -> Stage 2 resume + CSV export test), then set `SMOKE_TEST = False` and **Save Version -> Save & Run All (Commit)**.

In [ ]:
# ============================ 1. SETTINGS ============================
import os, sys, time, json, glob, shutil, subprocess

SESSION_START_TS = time.time()
os.environ["SESSION_START_TS"] = str(SESSION_START_TS)

SMOKE_TEST       = True   # True = fast Stage 1->2 resume + eval CSV smoke test; False = full 20k run
TIME_BUDGET_HRS  = 11.0
REPO_URL         = "https://github.com/AMYasserF/arams28k-baseline.git"
REPO_BRANCH      = "main"
DATASET_ZIP_URL  = "https://zenodo.org/records/21499649/files/AraMS-28k-HTR.zip?download=1"
HATFORMER_ZIP_URL = "https://zenodo.org/records/14165756/files/HATFormer.zip?download=1"
TOKENIZER_HF_REPO = "Archatext/hatformer-arams28k"
INIT_PATH_OVERRIDE = ""
ABLATION_ARGS    = ""     # e.g. "--no-film", "--no-multiscale", "--no-ctc", "--no-script-loss"

WORK = "/kaggle/working"
TMP  = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp/kaggle_temp"
os.makedirs(TMP, exist_ok=True)
REPO_DIR = f"{WORK}/arams28k-baseline"
RUN_DIR  = f"{WORK}/{'unified_smoke_run' if SMOKE_TEST else 'unified_htr_run'}"
SCRIPT   = f"{REPO_DIR}/scripts/models/unified_htr/train_unified.py"

def sh(cmd, check=True, cwd=None):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", cwd=cwd, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}")
    return rc

def find_files(root, name, max_depth=7):
    hits = []
    if not os.path.isdir(root): return hits
    base = root.rstrip("/").count("/")
    for dp, dn, fn in os.walk(root):
        if dp.count("/") - base >= max_depth: dn[:] = []
        if name in fn: hits.append(os.path.join(dp, name))
    return sorted(hits)

In [ ]:
# ============================ 2. GPU CHECK ============================
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
import torch, transformers, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU! Enable GPU T4 x2 in Session options."
print(f"GPU OK: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

In [ ]:
# ============================ 3. CLONE / UPDATE REPO & DEPS ============================
if os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git -C {REPO_DIR} fetch origin {REPO_BRANCH} && git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")
else:
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
sh("git log -1 --format='repo commit: %h  %s  (%cr)'", cwd=REPO_DIR)
assert os.path.isfile(SCRIPT), f"Missing {SCRIPT}"
sh("python -c 'import yaml, matplotlib, pandas, PIL, rapidfuzz, torchvision' 2>/dev/null || pip install -q pyyaml matplotlib pandas pillow rapidfuzz torchvision")

In [ ]:
# ============================ 4. DATA + PRETRAINED MUHARAF DECODER + TOKENIZER ============================
def locate_data():
    for root in ("/kaggle/input", f"{TMP}/data"):
        for m in find_files(root, "metadata.csv"):
            if os.path.isdir(os.path.join(os.path.dirname(m), "images")):
                return os.path.dirname(m)
    return None

DATA_DIR = locate_data()
if DATA_DIR is None:
    os.makedirs(f"{TMP}/data", exist_ok=True)
    sh(f"wget -nv -c -O {TMP}/AraMS-28k-HTR.zip '{DATASET_ZIP_URL}'")
    sh(f"unzip -q -o {TMP}/AraMS-28k-HTR.zip -d {TMP}/data && rm -f {TMP}/AraMS-28k-HTR.zip")
    DATA_DIR = locate_data()
assert DATA_DIR, "Could not locate AraMS-28k-HTR."
print("DATA_DIR =", DATA_DIR)

def model_dirs(root):
    out = []
    for cfgp in find_files(root, "config.json"):
        d = os.path.dirname(cfgp)
        if any(f.endswith((".safetensors", ".bin")) for f in os.listdir(d)):
            out.append(d)
    return out

if INIT_PATH_OVERRIDE:
    INIT_PATH = INIT_PATH_OVERRIDE
else:
    cands = [d for d in model_dirs("/kaggle/input") + model_dirs(f"{TMP}/hatformer") if "muharaf" in d.lower()]
    if not cands:
        os.makedirs(f"{TMP}/hatformer", exist_ok=True)
        sh(f"wget -nv -c -O {TMP}/HATFormer.zip '{HATFORMER_ZIP_URL}'")
        sh(f"unzip -q -o {TMP}/HATFormer.zip -d {TMP}/hatformer && rm -f {TMP}/HATFormer.zip")
        cands = [d for d in model_dirs(f"{TMP}/hatformer") if "muharaf" in d.lower()]
    assert cands, "hatformer-muharaf checkpoint folder not found; set INIT_PATH_OVERRIDE."
    INIT_PATH = cands[0]
print("INIT_PATH (hatformer-muharaf) =", INIT_PATH)

TOK_DST = f"{REPO_DIR}/scripts/models/hatformer/tokenizer/tokenizer.json"
if not os.path.isfile(TOK_DST) or os.path.getsize(TOK_DST) < 100_000:
    os.makedirs(os.path.dirname(TOK_DST), exist_ok=True)
    tok_cands = find_files("/kaggle/input", "tokenizer.json") + find_files(f"{TMP}/hatformer", "tokenizer.json") + find_files(INIT_PATH, "tokenizer.json", 1)
    tok_cands = [c for c in tok_cands if os.path.getsize(c) > 100_000]
    if tok_cands:
        shutil.copy(tok_cands[0], TOK_DST)
    else:
        from huggingface_hub import hf_hub_download
        shutil.copy(hf_hub_download(TOKENIZER_HF_REPO, "tokenizer.json", local_dir=f"{TMP}/tok_hf"), TOK_DST)
print("TOK_DST =", TOK_DST)

COMMON = f"--data-dir '{DATA_DIR}' --output-dir '{RUN_DIR}' --init-path '{INIT_PATH}' --tokenizer '{TOK_DST}' {ABLATION_ARGS}"

In [ ]:
# ============================ 5. PREFLIGHT ============================
sh(f"python -u {SCRIPT} {COMMON} --preflight --limit 200")

In [ ]:
# ============================ 6. TRAIN (STAGE 1 -> STAGE 2) ============================
if SMOKE_TEST:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    small = f"--limit 128 --stage1-steps 8 --val-every 4 --ckpt-every 4 --val-max-lines 16 --time-budget-hours 0 --resume '{RUN_DIR}'"
    sh(f"python -u {SCRIPT} {COMMON} --max-steps 6 {small}")
    print("\n==================== RESUME ACROSS STAGE 1 -> STAGE 2 ====================")
    sh(f"python -u {SCRIPT} {COMMON} --max-steps 16 {small}")
    st = json.load(open(f"{RUN_DIR}/status.json"))
    assert st["finished"] and st["step"] == 16 and st["stage"] == 2, st
    assert os.path.isfile(f"{RUN_DIR}/plots/training_curves.png")

    import pandas as pd
    smoke_preds = f"{RUN_DIR}/smoke_preds"
    shutil.rmtree(smoke_preds, ignore_errors=True)
    sh(f"python -u {SCRIPT} {COMMON} --eval-only --split test --limit 32 --num-beams 1 --checkpoint '{RUN_DIR}/best' --save-preds-dir '{smoke_preds}'")
    pcsvs = sorted(glob.glob(f"{smoke_preds}/unified_preds_book_*.csv"))
    assert len(pcsvs) > 0, f"No prediction CSVs in {smoke_preds}"
    for pcsv in pcsvs:
        d = pd.read_csv(pcsv)
        for col in ("line_uid", "book_id", "gt_normalized", "unified_pred", "ctc_pred"):
            assert col in d.columns, (pcsv, d.columns.tolist())
    assert os.path.isfile(f"{smoke_preds}/eval_summary.json")
    print("\nSMOKE TEST PASSED -> set SMOKE_TEST = False and Save & Run All (Commit).")
else:
    sh(f"python -u {SCRIPT} {COMMON} --resume auto --time-budget-hours {TIME_BUDGET_HRS}")

In [ ]:
# ============================ 7. CURVES ============================
from IPython.display import Image, display
curve = f"{RUN_DIR}/plots/training_curves.png"
if os.path.isfile(curve):
    display(Image(curve))
sh(f"tail -n 10 '{RUN_DIR}/metrics.csv'", check=False)
status = json.load(open(f"{RUN_DIR}/status.json")) if os.path.isfile(f"{RUN_DIR}/status.json") else {}
print("STATUS:", status)

In [ ]:
# ============================ 8. TEST EVALUATION ============================
if SMOKE_TEST:
    print("smoke test: skipping full test eval")
elif status.get("finished"):
    sh(f"python -u {SCRIPT} {COMMON} --eval-only --split test --num-beams 1 --checkpoint '{RUN_DIR}/best' --save-preds-dir '{RUN_DIR}/predictions_beams1'")
    sh(f"python -u {SCRIPT} {COMMON} --eval-only --split test --num-beams 3 --checkpoint '{RUN_DIR}/best' --save-preds-dir '{RUN_DIR}/predictions_beams3'")
    for bdir in (f"{RUN_DIR}/predictions_beams1", f"{RUN_DIR}/predictions_beams3"):
        print("\n=== " + bdir + " ===")
        print(open(f"{bdir}/eval_summary.json", encoding="utf-8").read())
else:
    print(f"Paused at step {status.get('step')} (Stage {status.get('stage')}) — attach output dataset & resume.")

In [ ]:
# ============================ 9. WRAP-UP ============================
if "status" not in globals():
    status = json.load(open(f"{RUN_DIR}/status.json")) if os.path.isfile(f"{RUN_DIR}/status.json") else {}
if not SMOKE_TEST and status.get("finished") and os.path.isdir(f"{RUN_DIR}/best"):
    zip_path = shutil.make_archive(f"{WORK}/unified_best_weights", "zip", root_dir=f"{RUN_DIR}/best")
    print(f"[WRAP-UP] Created self-contained archive: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if SMOKE_TEST:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
sh(f"du -sh {WORK}/* 2>/dev/null | sort -h | tail", check=False)